In [0]:
from __future__ import annotations
import os, sys, time, json, traceback, tempfile, joblib
from typing import Any, Dict
import mlflow
import mlflow.sklearn

In [0]:
sys.path.append("/Workspace/9900-f18a-cake")
sys.path.append("/Workspace/9900-f18a-cake/mt-method2/src")
sys.path.append("/Workspace/9900-f18a-cake/mt-method2")
from mch.models.training import BatchModelTrainer

In [0]:
def _norm_exp(value: str | None) -> str | None:
    """Return a valid experiment path or None if value is empty/'none'."""
    if value is None:
        return None
    v = str(value).strip()
    if v == "" or v.lower() == "none":
        return None
    return v

In [0]:
def _bool_env(key: str, default: bool) -> bool:
    v = os.getenv(key)
    if v is None:
        return default
    return str(v).strip().lower() in {"1", "true", "yes", "y"}

In [0]:
def _int_env(key: str, default: int) -> int:
    v = os.getenv(key)
    return int(v) if v is not None and str(v).strip() != "" else default

In [0]:
def _set_env_for_trainer(node_id: str) -> Dict[str, Any]:
    """Forward env expected by your trainer; enforce single-node via MCH_ONLY_NODE."""
    env_applied = {
        "MCH_ONLY_NODE": node_id,
        "MCH_SAVE_MODE": "nosave",  # we’ll persist via MLflow instead
        "MCH_DISABLE_DM": os.getenv("DISABLE_DM", "1"),
        "MCH_PREFILTER_TOPK": str(_int_env("PREFILTER_TOPK", 200)),
        "MCH_PREFILTER_SCAN_MAX": str(_int_env("PREFILTER_SCAN_MAX", 20000)),
        "MCH_PREFILTER_CHUNK_SIZE": str(_int_env("PREFILTER_CHUNK_SIZE", 5000)),
        "MCH_RF_N_JOBS": str(_int_env("RF_N_JOBS", 1)),
        "MCH_CV_N_JOBS": str(_int_env("CV_N_JOBS", 1)),
    }
    for k, v in env_applied.items():
        os.environ[k] = str(v)
    return env_applied

In [0]:
def _extract_node_stats(stats: Dict[str, Any], node_id: str) -> Dict[str, Any]:
    if not isinstance(stats, dict): return {}
    if node_id in stats: return stats.get(node_id, {})
    if len(stats) == 1: return list(stats.values())[0]
    return {}

In [0]:
def _try_log_model(node_stats: Dict[str, Any], node_id: str) -> None:
    """If trainer exposes a fitted estimator, log it to MLflow."""
    import mlflow.sklearn
    for key in ("estimator", "model", "pipeline"):
        est = node_stats.get(key)
        if est is None: 
            continue
        try:
            mlflow.sklearn.log_model(est, artifact_path=f"model/{node_id}")
            print(f"[MLflow] Logged model at model/{node_id}")
            return
        except Exception as e:
            print(f"[WARN] Could not log model from node_stats['{key}']: {e}")
    print("[INFO] No fitted estimator found in node_stats; skipped model logging.")

In [0]:
def _print_result_summary(result: Dict[str, Any], default_node_id: str = "") -> None:
    node = result.get("node_id") or default_node_id or "UNKNOWN"
    print(f"\n=== RESULTS for node: {node} ===")
    if result.get("error"):
        print("Status : FAILED")
        print("Error  :", result["error"])
        if result.get("trace"):
            print("\nTraceback:")
            print(result["trace"])
    else:
        print("Status : OK")
        metrics = result.get("metrics") or {}
        if metrics:
            w = max(len(k) for k in metrics.keys())
            for k in sorted(metrics.keys()):
                v = metrics[k]
                try:
                    print(f"{k.rjust(w)} : {float(v):.4f}")
                except Exception:
                    print(f"{k.rjust(w)} : {v}")
        else:
            print("No metrics were logged (likely skipped due to sample count or single subgroup).")

    if result.get("t_sec") is not None:
        print(f"Elapsed (sec) : {float(result['t_sec']):.2f}")

    try:
        run = mlflow.active_run()
        if run:
            print("MLflow run id :", run.info.run_id)
    except Exception:
        pass

In [0]:
#NODE_ID = os.getenv("NODE_ID", "").strip()
NODE_ID = "Haematological malignancy"
if not NODE_ID:
    raise ValueError("NODE_ID is required (e.g., NODE_ID=ZERO2)")

# Optional MLflow config
EXPERIMENT_PATH = _norm_exp(os.getenv("MLFLOW_EXPERIMENT"))
if EXPERIMENT_PATH is None:
    EXPERIMENT_PATH = "/Workspace/9900-f18a-cake/classifier"  # default

PARENT_RUN_ID = _norm_exp(os.getenv("PARENT_RUN_ID"))  # optional; used as tag only

print("Using configuration:")
print("  NODE_ID         :", NODE_ID)
print("  EXPERIMENT_PATH :", EXPERIMENT_PATH)
print("  PARENT_RUN_ID   :", PARENT_RUN_ID or "(none)")

In [0]:
# Create/activate experiment by NAME (no experiment_id API here)
mlflow.set_experiment(EXPERIMENT_PATH)
# Confirm effective experiment
active_exp = mlflow.get_experiment_by_name(EXPERIMENT_PATH)
print("Active experiment:", active_exp.name, active_exp.experiment_id)

In [0]:
forwarded_env = _set_env_for_trainer(NODE_ID)

result = {
    "ok": False,
    "node_id": NODE_ID,
    "metrics": {},
    "error": None,
    "trace": None,
    "t_sec": None,
}

t0 = time.time()

with mlflow.start_run(run_name=f"child-{NODE_ID}") as run:
    if PARENT_RUN_ID:
        mlflow.set_tag("mlflow.parentRunId", PARENT_RUN_ID)

    mlflow.set_tags({
        "orchestrator": "databricks_job_notebook",
        "node_id": NODE_ID,
        "only_node": NODE_ID,
        "codepath": "Child4_nb",
    })
    mlflow.log_dict(forwarded_env, artifact_file="child_env.json")

    try:
        print(f"==== Training node_id={NODE_ID} ====")
        trainer = BatchModelTrainer()
        stats_all = trainer.train_all_models(save_dir=None, raise_on_error=False)
        safe_node = NODE_ID.replace(" ", "_").replace("/", "_")

        node_key = NODE_ID
        node_stats = stats_all.get(node_key, {})

        # 1) Log the model if present
        est = node_stats.get("estimator")
        if est is None:
            est = getattr(trainer, "models", {}).get(node_key)

        #print("DEBUG: keys in node_stats:", list(node_stats.keys()))
        if est is not None:
            mlflow.sklearn.log_model(est, artifact_path=f"models/{node_key.replace(' ','_')}")
            mlflow.sklearn.log_model(est, artifact_path=f"models/{safe_node}")
            with tempfile.TemporaryDirectory() as td:
                jp = os.path.join(td, f"{safe_node}.joblib")
                joblib.dump(est, jp)
                mlflow.log_artifact(jp, artifact_path=f"joblib/{safe_node}")
        else:
            print("No estimator found; nothing to log.")

        # 2) Log metrics (optional but nice)
        for k, v in (node_stats.get("metrics") or {}).items():
            try:
                mlflow.log_metric(k, float(v))
            except Exception:
                pass

        result["ok"] = True
        result["metrics"] = metrics

    except Exception as e:
        result["error"] = f"{type(e).__name__}: {e}"
        result["trace"] = traceback.format_exc()

    finally:
        result["t_sec"] = round(time.time() - t0, 3)
        _print_result_summary(result, default_node_id=NODE_ID)